# EDI Remaining-Method Protocol and Map-Pair Audit v1
Run all cells in a **CPU Colab runtime**. This notebook audits Eigen-CAM, Integrated Gradients, Score-CAM and SHAP using existing files in Drive. It refreshes candidate lookup while preserving `_downsampled`, fingerprints arrays, compares native-grid diagnostic metrics and (when an independently documented protocol exists) reproduces recorded metrics under that protocol.

No training, map generation, canonical integration or source edits occur. Completion of this notebook means the audit executed, not that every comparison passed. Missing protocols/maps remain explicit outcomes. Upload the printed compact ZIP after completion. Completed rows are checkpointed and resumed on rerun.

This is the next audit stage, not a substitute for the later external-corpus, classification, and table/figure lineage audits. Repository preparation exports evidence indexes and a draft layout; nothing is published to GitHub.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, zipfile, re, time, platform, sys
import numpy as np
import pandas as pd
import scipy, skimage
from skimage.metrics import structural_similarity
from functools import lru_cache
from IPython.display import display
ROOT=Path('/content/drive/MyDrive/Dissertation')
RUN_ID='edi_remaining_method_protocol_pair_audit_v1'
OUT=ROOT/'provenance_inventory'/RUN_ID
TABLE=OUT/'tables'; TABLE.mkdir(parents=True,exist_ok=True)
PREV=ROOT/'provenance_inventory/edi_internal_attribution_provenance_integration_v1'
REC=ROOT/'provenance_inventory/edi_gradcam_remediation_provenance_recovery_v1'
FOLDER=ROOT/'provenance_inventory/edi_dissertation_folder_artifact_map_v1'
SOURCE=PREV/'tables/internal_manifest_provenance_corrected.csv'
INVENTORY=REC/'tables/table_map_inventory.csv'
PROTOCOL_FILE=OUT/'protocol_registry.json'
K=['experiment_id','image_id','preprocess','resolution','attribution_method']
METHODS=['eigencam','ig','scorecam','shap']
TOL=1e-4
MAX_NOTEBOOK_BYTES=20_000_000
MAX_NOTEBOOKS=100

def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def atomic_json(p,x):
 p=Path(p);t=p.with_name(p.name+'.tmp');t.write_text(json.dumps(x,indent=2,allow_nan=False));os.replace(t,p)
def atomic_csv(p,x):
 p=Path(p);t=p.with_name(p.name+'.tmp');x.to_csv(t,index=False);os.replace(t,p)
def read(p):
 if not Path(p).is_file():raise FileNotFoundError(f'Required exact source missing: {p}')
 return pd.read_csv(p,low_memory=False)
def flag(x):return str(x).lower() in ('true','1','1.0')
def iid(x):return str(x).strip().upper().replace('-','_')
def method(x):
 x=str(x).lower().replace('_','').replace('-','')
 return {'integratedgradients':'ig','integratedgradient':'ig'}.get(x,x)
def parse_map(path):
 p=Path(path)
 mt=None
 for part in reversed(p.parts):
  q=method(part)
  if q in METHODS:mt=q;break
 if mt is None:
  low=p.name.lower().replace('_','').replace('-','')
  mt=next((m for m in ['eigencam','scorecam','shap'] if m in low),None)
  if mt is None and re.search(r'_ig_',p.name,re.I):mt='ig'
 im=re.search(r'ISIC[_-]?\d+(?:_downsampled)?',p.name,re.I)
 exp=next((x for x in reversed(p.parts) if re.search(r'_r(?:128|160|192|224)_',x)),None)
 return (exp,mt,iid(im.group()) if im else None)
A=read(SOURCE)
A['source_image_id']=A.image_id.astype(str);A.image_id=A.image_id.map(iid)
A.resolution=pd.to_numeric(A.resolution,errors='raise').astype(int)
A.attribution_method=A.attribution_method.map(method)
if len(A)!=23040 or A.duplicated(K).any():raise ValueError('Internal source row/key gate failed')
status=json.loads((PREV/'final_status.json').read_text())
if not status.get('cam_provenance_integration_authorized'):raise ValueError('Prior CAM integration not authorized')
D=A[A.attribution_method.isin(METHODS)].copy()
if len(D)!=15360:raise ValueError('Remaining-method row gate failed')
SOURCE_HASHES={str(p):sha(p) for p in [SOURCE,INVENTORY,PREV/'final_status.json']}
ENV={'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pandas':pd.__version__,'scipy':scipy.__version__,'skimage':skimage.__version__}
print('Remaining comparisons:',len(D));display(pd.Series(ENV))

## Recover protocol evidence without guessing
The bounded folder inventory is used to locate candidate generation/analysis notebooks. Relevant cells are copied as text; they are **never executed**. If the notebooks were not stored in this Drive folder, this step records that limitation.

Optional `protocol_registry.json` entries must be supported by the exact original code and its SHA-256. The registry specifies the comparison protocol and is not automatically created from a numerical match. Without it, native-grid results are diagnostic only. The README template explains the schema. This notebook supports raw or independent min–max normalization, native/explicit/input-resolution grids, nearest/linear/cubic resizing, and explicit SSIM parameters. Unsupported original protocols require a versioned adapter rather than substitution.

In [ ]:
evidence=[]; failures=[]
fi=FOLDER/'tables/table_file_inventory.csv'
if fi.is_file():
 files=read(fi)
 paths=sorted(set(files.loc[files.path.astype(str).str.endswith('.ipynb'), 'path'].astype(str)))
 priority=lambda p: (not bool(re.search(r'edi|attribution|p5|p6|eigencam|scorecam|shap|integrated',p,re.I)),p)
 paths=sorted(paths,key=priority)
 for path in paths[:MAX_NOTEBOOKS]:
  try:
   p=Path(path)
   if p.stat().st_size>MAX_NOTEBOOK_BYTES:raise ValueError('Notebook exceeds byte limit')
   obj=json.loads(p.read_text());digest=sha(p)
   for i,c in enumerate(obj.get('cells',[])):
    src=c.get('source',[]);src=src if isinstance(src,str) else ''.join(src)
    if c.get('cell_type')=='code' and re.search(r'structural_similarity|pearson|integrated.?gradient|score.?cam|eigen.?cam|shap|ssim',src,re.I):
     evidence.append({'path':path,'sha256':digest,'cell_index':i,'source':src})
  except Exception as ex:failures.append({'path':path,'error':str(ex)})
else:
 paths=[];failures.append({'path':str(fi),'error':'Folder inventory absent; map audit continues'})
atomic_csv(TABLE/'table_protocol_code_evidence.csv',pd.DataFrame(evidence,columns=['path','sha256','cell_index','source']))
atomic_csv(TABLE/'table_protocol_scan_errors.csv',pd.DataFrame(failures,columns=['path','error']))
atomic_json(OUT/'protocol_scan_status.json',{'candidate_notebooks':len(paths),'attempted':min(len(paths),MAX_NOTEBOOKS),'bounded':True,'cells_found':len(evidence),'no_code_executed':True})
registry=json.loads(PROTOCOL_FILE.read_text()) if PROTOCOL_FILE.is_file() else {'protocols':[]}
protocols={}
for q in registry.get('protocols',[]):
 key=(q['configuration'],method(q['method']),int(q['resolution']))
 if key in protocols:raise ValueError('Duplicate protocol scope')
 p=Path(q['evidence_path'])
 if not p.is_file() or sha(p)!=q['evidence_sha256']:raise ValueError('Protocol evidence hash gate failed')
 if q.get('evidence_reviewed') is not True:raise ValueError('Protocol must be reviewed against original code')
 for f in ['grid','normalization','ssim_win_size','ssim_data_range','gaussian_weights','sigma','use_sample_covariance','evidence_locator','generation_protocol_notes']:
  if f not in q:raise ValueError('Missing protocol field: '+f)
 if q['grid'] not in ['native','input_resolution','explicit']:raise ValueError('Unsupported grid')
 if q['normalization'] not in ['raw','minmax']:raise ValueError('Unsupported normalization')
 if q['grid']!='native' and q.get('interpolation') not in ['nearest','linear','cubic']:raise ValueError('Unsupported interpolation')
 if q['grid']=='explicit' and len(q.get('shape',[]))!=2:raise ValueError('Explicit shape required')
 protocols[key]=q
print('Protocol evidence cells:',len(evidence),'reviewed protocol scopes:',len(protocols))

In [ ]:
inv=read(INVENTORY);lookup={}
for path in inv.path.dropna().astype(str):
 exp,mt,im=parse_map(path)
 if exp and mt and im:lookup.setdefault((exp,mt,im),set()).add(path)
lut=read(ROOT/'provenance_inventory/edi_internal_baseline_pairing_map_provenance_audit_v1/tables/table_intended_baseline_lookup.csv')
if lut.duplicated(['configuration','resolution']).any() or not lut.unambiguous.map(flag).all():raise ValueError('Baseline lookup is ambiguous')
base={(x.configuration,int(x.resolution)):x.intended_baseline_id for x in lut.itertuples()}
plan=[]
for row in D.to_dict('records'):
 cfg=next((c for c in ['train_ham_test_isic','combined','ham','isic'] if row['experiment_id'].startswith(c+'_')),None)
 intended=base.get((cfg,row['resolution']))
 if not intended:raise ValueError('Unresolved intended baseline')
 cp=set(lookup.get((row['experiment_id'],row['attribution_method'],row['image_id']),set()))
 recorded=str(row.get('comparison_heatmap_npy',''))
 if parse_map(recorded)==(row['experiment_id'],row['attribution_method'],row['image_id']):cp.add(recorded)
 bp=sorted(lookup.get((intended,row['attribution_method'],row['image_id']),set()))
 plan.append({**{k:row[k] for k in K},'source_image_id':row['source_image_id'],'configuration':cfg,'recorded_baseline_id':row['baseline_experiment_id'],'intended_baseline_id':intended,'baseline_id_mismatch':row['baseline_experiment_id']!=intended,'baseline_candidates':json.dumps(bp),'comparison_candidates':json.dumps(sorted(cp)),'baseline_candidate_count':len(bp),'comparison_candidate_count':len(cp)})
P=pd.DataFrame(plan)
atomic_csv(TABLE/'table_refreshed_pair_plan.csv',P)
display(P.groupby('attribution_method').agg(rows=('image_id','size'),with_baseline=('baseline_candidate_count',lambda x:int((x>0).sum())),ambiguous_baseline=('baseline_candidate_count',lambda x:int((x>1).sum()))))
atomic_csv(TABLE/'table_image_id_aliases.csv',A[['source_image_id','image_id']].drop_duplicates())

## Resume-safe existing-array audit
Arrays are hashed and inspected once per path. Only finite, nonconstant two-dimensional arrays are compared. Multiple paths are accepted as one array only when their byte hashes agree; conflicting arrays remain ambiguous. SHA-256 identifies bytes, not the model that produced them.

Native-grid diagnostic checks use independent min–max normalization and SSIM window 7/data range 1. These are labeled assumptions. Documented-protocol checks are distinct. For mismatched reference models, the notebook exports newly calculated intended-pair metrics separately; it does not expect them to equal the historical metrics. Checkpoints are bound to source, registry and audit-code hashes.

In [ ]:
# Array metadata and metrics; no attribution libraries or model loading needed.
import cv2
ENV['opencv']=cv2.__version__
@lru_cache(maxsize=256)
def array(path):
 x=np.load(path,allow_pickle=False)
 if not isinstance(x,np.ndarray) or x.ndim!=2:raise ValueError('Expected 2D ndarray; no implicit squeeze or reduction')
 x=x.astype(np.float64)
 if not np.isfinite(x).all():raise ValueError('Nonfinite array')
 if np.ptp(x)==0:raise ValueError('Constant array: Pearson undefined')
 return x
fingerprints={}
def inspect(path):
 if path not in fingerprints:
  try:
   p=Path(path);h=sha(p);x=np.load(p,allow_pickle=False)
   fingerprints[path]={'path':path,'sha256':h,'bytes':p.stat().st_size,'shape':json.dumps(list(x.shape)),'finite':bool(np.isfinite(x).all()),'constant':bool(np.ptp(x)==0),'error':''}
  except Exception as ex:fingerprints[path]={'path':path,'error':str(ex)}
 return fingerprints[path]
def select(paths):
 paths=json.loads(paths);info=[inspect(p) for p in paths]
 if not info:return None,'missing'
 good=[q for q in info if not q.get('error')]
 if len(good)!=len(info):return None,'candidate_unreadable'
 if len({q['sha256'] for q in good})!=1:return None,'ambiguous_content'
 return paths[0],'unique_or_byte_identical'
def metrics(bp,cp,q,res):
 a=array(bp).copy();b=array(cp).copy()
 grid=q['grid']
 if grid!='native':
  shape=[res,res] if grid=='input_resolution' else q['shape']
  inter={'nearest':cv2.INTER_NEAREST,'linear':cv2.INTER_LINEAR,'cubic':cv2.INTER_CUBIC}[q['interpolation']]
  a=cv2.resize(a,(int(shape[1]),int(shape[0])),interpolation=inter);b=cv2.resize(b,(int(shape[1]),int(shape[0])),interpolation=inter)
 if a.shape!=b.shape:raise ValueError('Grid shape mismatch')
 if q['normalization']=='minmax':
  a=(a-a.min())/np.ptp(a);b=(b-b.min())/np.ptp(b)
 if np.ptp(a)==0 or np.ptp(b)==0:raise ValueError('Transformed array constant')
 w=int(q['ssim_win_size'])
 if w<3 or w%2==0 or min(a.shape)<w:raise ValueError('Invalid SSIM window for map shape')
 dr=float(q['ssim_data_range'])
 if dr<=0:raise ValueError('SSIM data range must be positive')
 pear=float(np.corrcoef(a.ravel(),b.ravel())[0,1])
 ss=float(structural_similarity(a,b,win_size=w,data_range=dr,gaussian_weights=bool(q['gaussian_weights']),sigma=float(q['sigma']),use_sample_covariance=bool(q['use_sample_covariance'])))
 edi=((1-(pear+1)/2)+(1-ss))/2
 if not np.isfinite([pear,ss,edi]).all():raise ValueError('Nonfinite metrics')
 return {'pearson_corr':pear,'ssim':ss,'edi_primary':edi}
DIAG={'grid':'native','normalization':'minmax','ssim_win_size':7,'ssim_data_range':1,'gaussian_weights':False,'sigma':1.5,'use_sample_covariance':True}


In [ ]:
AUDIT_CODE_HASH=hashlib.sha256((inspect.__code__.co_code+select.__code__.co_code+metrics.__code__.co_code+parse_map.__code__.co_code)+repr((inspect.__code__.co_consts,select.__code__.co_consts,metrics.__code__.co_consts,parse_map.__code__.co_consts)).encode()).hexdigest()
run_signature=hashlib.sha256(json.dumps({'sources':SOURCE_HASHES,'registry':registry,'diagnostic':DIAG,'tol':TOL,'code_hash':AUDIT_CODE_HASH,'environment':ENV},sort_keys=True).encode()).hexdigest()
config_path=OUT/'run_config.json'
if config_path.is_file() and json.loads(config_path.read_text())['signature']!=run_signature:
 raise ValueError('Inputs/settings changed: choose a new RUN_ID to preserve checkpoint evidence')
atomic_json(config_path,{'signature':run_signature,'sources':SOURCE_HASHES,'registry':registry,'environment':ENV,'diagnostic_protocol':DIAG,'tolerance':TOL,'audit_code_hash':AUDIT_CODE_HASH})
checkpoint=TABLE/'table_pair_audit.csv'
results=read(checkpoint).to_dict('records') if checkpoint.is_file() else []
done={r['row_id'] for r in results}
oldfp=TABLE/'table_map_fingerprints.csv'
if oldfp.is_file():
 for q in read(oldfp).fillna('').to_dict('records'):fingerprints[q['path']]=q
src={tuple(r[k] for k in K):r for r in D.to_dict('records')}
started=time.time()
for idx,row in enumerate(P.to_dict('records')):
 rid=hashlib.sha256(json.dumps([str(row[k]) for k in K]).encode()).hexdigest()
 if rid in done:continue
 out={**{k:row[k] for k in K},'row_id':rid,'configuration':row['configuration'],'baseline_id_mismatch':bool(row['baseline_id_mismatch']),'protocol_documented':False,'recorded_pair_reproduced':False,'intended_pair_calculated':False}
 s=src[tuple(row[k] for k in K)]
 bp,bs=select(row['baseline_candidates']);cp,cs=select(row['comparison_candidates'])
 out.update(intended_baseline_path=bp or '',comparison_path=cp or '',baseline_selection=bs,comparison_selection=cs)
 q=protocols.get((row['configuration'],row['attribution_method'],row['resolution']))
 out['protocol_documented']=q is not None
 if bp and cp:
  try:
   m=metrics(bp,cp,q or DIAG,row['resolution']);out.update({'intended_'+k:v for k,v in m.items()});out['intended_pair_calculated']=True
   out['intended_metric_basis']='documented_protocol' if q else 'native_diagnostic_assumptions'
  except Exception as ex:out['intended_error']=str(ex)
 rb=str(s.get('baseline_heatmap_npy',''));rc=str(s.get('comparison_heatmap_npy',''))
 out['recorded_baseline_path']=rb;out['recorded_comparison_path']=rc
 try:
  inspect(rb);inspect(rc)
  m=metrics(rb,rc,q or DIAG,row['resolution'])
  for key,val in m.items():out['recorded_'+key]=val;out[key+'_error']=abs(val-float(s[key]))
  out['recorded_numeric_match']=all(out[k+'_error']<=TOL for k in m)
  out['recorded_pair_reproduced']=bool(q and out['recorded_numeric_match'])
 except Exception as ex:out['recorded_error']=str(ex)
 out['status']='documented_intended_pair_calculated' if q and out['intended_pair_calculated'] else 'unresolved_protocol_or_pair'
 results.append(out);done.add(rid)
 if len(results)%100==0:
  atomic_csv(checkpoint,pd.DataFrame(results));atomic_csv(oldfp,pd.DataFrame(fingerprints.values()))
  atomic_json(OUT/'progress.json',{'processed':len(results),'expected':len(P),'elapsed_seconds':time.time()-started})
  print(f'{len(results):,}/{len(P):,} comparisons; {len(fingerprints):,} map fingerprints',flush=True)
atomic_csv(checkpoint,pd.DataFrame(results));atomic_csv(oldfp,pd.DataFrame(fingerprints.values()))
Result=pd.DataFrame(results)
if len(Result)!=15360 or Result.row_id.duplicated().any():raise ValueError('Completion/key gate failed')
print('Pair audit complete. Unresolved outcomes are retained.')

In [ ]:
Summary=Result.groupby('attribution_method').agg(rows=('row_id','size'),protocol_documented=('protocol_documented',lambda s:sum(map(flag,s))),recorded_pair_reproduced=('recorded_pair_reproduced',lambda s:sum(map(flag,s))),intended_pair_calculated=('intended_pair_calculated',lambda s:sum(map(flag,s))))
atomic_csv(TABLE/'table_method_audit_summary.csv',Summary.reset_index());display(Summary)
unchanged=all(sha(p)==h for p,h in SOURCE_HASHES.items())
scientific=bool(unchanged and Result.protocol_documented.map(flag).all() and Result.intended_pair_calculated.map(flag).all() and not Result.baseline_id_mismatch.map(flag).any() and Result.recorded_pair_reproduced.map(flag).all())
final={'completed_utc':datetime.now(timezone.utc).isoformat(),'audit_execution_complete':True,'comparisons_inspected':len(Result),'source_files_unchanged':unchanged,'remaining_method_canonical_reproduction_passed':scientific,'canonical_integration_authorized':False,'full_corpus_integration_authorized':False,'models_retrained':False,'maps_generated':False,'scope':'Remaining-method existing-array and documented-protocol audit; diagnostic assumptions are not provenance validation'}
atomic_json(OUT/'final_status.json',final)
# Future repository assets: no image/model/Drive data are copied or published.
repo=OUT/'repository_preparation';repo.mkdir(exist_ok=True)
(repo/'README_repository_plan.md').write_text('''# Dissertation repository preparation

Draft layout: notebooks/audits; src/metrics; configs; docs/provenance; tests; evidence/public_summaries.

Keep patient/image data, model weights, maps, Drive IDs, private paths and generated bulk outputs outside git. Provide verified dataset access instructions, task-specific configuration records and compact evidence summaries. Select the repository name and license with the author before publication.

The audit outputs are working evidence, not automatically public artifacts. Before release, reconcile source hashes, remove private paths from public summaries, document the original attribution-generation implementation, and audit the combined corpus plus every reported table/figure. No GitHub repository was created by this notebook.
''')
(repo/'.gitignore').write_text('data/\nmodels/\nheatmaps/\noutputs/\n*.keras\n*.h5\n*.npy\n*.npz\n.ipynb_checkpoints/\n__pycache__/\n')
artifacts=[]
for p in sorted(OUT.rglob('*')):
 if p.is_file() and p.name!='table_output_artifact_index.csv':artifacts.append({'relative_path':str(p.relative_to(OUT)),'sha256':sha(p),'bytes':p.stat().st_size})
atomic_csv(TABLE/'table_output_artifact_index.csv',pd.DataFrame(artifacts))
archive=OUT.parent/(RUN_ID+'_compact_bundle.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for p in sorted(OUT.rglob('*')):
  if p.is_file() and not p.name.endswith('.tmp'):z.write(p,p.relative_to(OUT))
display(pd.Series(final));print('Upload this ZIP:',archive)
if not unchanged:raise ValueError('Source preservation failed')

## Protocol registry schema (optional; use only after reviewing original source)
Place `protocol_registry.json` in this run's output folder **before first execution** if you already have reviewed protocol evidence. Otherwise run without it and use the exported code evidence to prepare a new versioned audit. Never fill a registry merely because tested settings matched stored numbers.

```json
{"protocols": [{
 "configuration": "ham", "method": "eigencam", "resolution": 128,
 "evidence_path": "/content/drive/MyDrive/Dissertation/EXACT_ORIGINAL_NOTEBOOK.ipynb",
 "evidence_sha256": "ACTUAL_SHA256", "evidence_reviewed": true,
 "evidence_locator": "cell index and function names",
 "generation_protocol_notes": "Actual model/target/layer and method-specific settings, normalization, versions and limitations",
 "grid": "native", "normalization": "minmax", "ssim_win_size": 7,
 "ssim_data_range": 1.0, "gaussian_weights": false, "sigma": 1.5,
 "use_sample_covariance": true
}]}
```
These example settings illustrate syntax and are not verified historical settings. Changed inputs or protocols require a new RUN_ID; original results remain preserved. Array-level reproduction still does not independently prove the attribution method was correctly implemented or the arrays came from the intended checkpoint; generation/model provenance requires its own evidence.
